# Tutorial 1: Data Loading and Exploration

## Overview

This notebook demonstrates how to load and explore the heterogeneous data sources used in Fioracle.

### Data Sources (1945-2025)

| Source | Frequency | Coverage | Features |
|--------|-----------|----------|----------|
| **GPR** (Geopolitical Risk) | Daily | 1985-2025 | Risk index |
| **EPU** (Policy Uncertainty) | Daily | 1985-2025 | Uncertainty index |
| **Shiller** | Monthly | 1871-2023 | SP500, yields, earnings |
| **JST** (Macrohistory) | Annual | 1870-2017 | Macro-financial data |
| **EFW** (Economic Freedom) | Annual | 1970-2023 | Freedom indices |
| **KOF** (Globalization) | Annual | 1970-2024 | Globalization indices |
| **FRED-MD** | Monthly | 1959-2025 | 127 macro series |
| **LSEG** (Modern Data) | Daily | 2000-2025 | ETFs, yields, VIX |

### Learning Objectives

1. Understand Fioracle's data architecture
2. Use `DataPipeline` class for unified data loading
3. Explore aligned daily dataset
4. Visualize data coverage and quality
5. Understand frequency alignment strategies

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 50)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ Imports successful")

## 2. Using DataPipeline Class

The `DataPipeline` class provides unified access to all data sources with:
- Automatic frequency alignment (annual/monthly/daily → daily)
- Smart parquet caching for performance
- Two modes: 'basic' (20 features) and 'full' (90+ features)

In [ ]:
from src.core.data import DataPipeline

# Initialize pipeline in basic mode
pipeline = DataPipeline(mode='basic', cache_dir='../data/cache')

print("✓ DataPipeline initialized")
print(f"  Mode: {pipeline.mode}")
print(f"  Cache directory: {pipeline.cache_dir}")

### Load Data for 1985-2010 Period

In [ ]:
# Load and align data
data = pipeline.load(
    start_date='1985-01-01',
    end_date='2010-12-31',
    force_reload=False  # Use cache if available
)

print(f"\n✓ Loaded data: {data.shape}")
print(f"  Date range: {data.index.min()} to {data.index.max()}")
print(f"  Business days: {len(data)}")
print(f"  Features: {len(data.columns)}")

## 3. Explore the Dataset

In [ ]:
# View first few rows
print("First 5 rows:")
data.head()

In [ ]:
# Summary statistics
print("Summary Statistics:")
data.describe()

In [ ]:
# Check for missing values
missing_pct = (data.isna().sum() / len(data) * 100).sort_values(ascending=False)

print("Missing Data by Feature:")
print(missing_pct.head(10))

# Visualize missing data
fig, ax = plt.subplots(figsize=(12, 6))
missing_pct.head(15).plot(kind='bar', ax=ax, color='coral')
ax.set_title('Top 15 Features by Missing Data (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Missing Data (%)')
ax.set_xlabel('Feature')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Visualize Data Coverage

Understand temporal coverage of each data source.

In [ ]:
# Create coverage heatmap
def plot_data_coverage(df, sample_freq='M'):
    """
    Plot data availability heatmap.
    
    Args:
        df: DataFrame to analyze
        sample_freq: Sampling frequency for visualization ('M'=monthly, 'Q'=quarterly)
    """
    # Sample data at lower frequency for visualization
    sampled = df.resample(sample_freq).first()
    
    # Create binary availability matrix (1 = data available, 0 = missing)
    availability = (~sampled.isna()).astype(int)
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(16, 8))
    
    sns.heatmap(
        availability.T,
        cmap='RdYlGn',
        cbar_kws={'label': 'Data Available'},
        ax=ax,
        xticklabels=50  # Show every 50th date
    )
    
    ax.set_title('Data Coverage Heatmap (1=Available, 0=Missing)', 
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Feature')
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    coverage_pct = (availability.sum() / len(availability) * 100).sort_values(ascending=False)
    print(f"\nCoverage Summary (n={len(availability)} {sample_freq} periods):")
    print(f"  Mean coverage: {coverage_pct.mean():.1f}%")
    print(f"  Median coverage: {coverage_pct.median():.1f}%")
    print(f"  Features with 100% coverage: {(coverage_pct == 100).sum()}")
    print(f"  Features with <50% coverage: {(coverage_pct < 50).sum()}")

# Plot coverage
plot_data_coverage(data, sample_freq='Q')  # Quarterly for readability

## 5. Visualize Key Time Series

Plot important features to understand their dynamics.

In [ ]:
# Select key features to plot
key_features = [
    'macro_gpr',      # Geopolitical Risk
    'macro_epu',      # Economic Policy Uncertainty
    'shiller_sp500',  # S&P 500
    'shiller_gs10',   # 10-Year Treasury Yield
]

# Filter available features
available_features = [f for f in key_features if f in data.columns]

if len(available_features) > 0:
    fig, axes = plt.subplots(len(available_features), 1, figsize=(14, 3*len(available_features)))
    
    if len(available_features) == 1:
        axes = [axes]
    
    for i, feature in enumerate(available_features):
        ax = axes[i]
        data[feature].plot(ax=ax, color='steelblue', linewidth=1)
        ax.set_title(f'{feature}', fontsize=12, fontweight='bold')
        ax.set_xlabel('Date')
        ax.grid(alpha=0.3)
        
        # Shade recession periods (simplified - major recessions)
        recessions = [
            ('1990-07-01', '1991-03-31'),  # 1990 recession
            ('2001-03-01', '2001-11-30'),  # 2001 recession
            ('2007-12-01', '2009-06-30'),  # Great Recession
        ]
        
        for start, end in recessions:
            ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), 
                      alpha=0.2, color='red', label='Recession')
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠ No key features found in dataset")

## 6. Understanding Frequency Alignment

Fioracle aligns multi-frequency data using intelligent strategies:

### Annual → Daily
- **Policy variables** (EFW, KOF): Step function (constant within year)
- **Macro variables** (JST): Linear interpolation

### Monthly → Daily
- **Yields/Prices**: PCHIP interpolation (smooth curves)
- **Categorical**: Forward-fill

### Daily → Daily
- Direct alignment with forward-fill for missing business days

In [ ]:
# Demonstrate alignment for different frequencies
import matplotlib.dates as mdates

# Find features from different sources
gpr_feature = 'macro_gpr' if 'macro_gpr' in data.columns else None
shiller_feature = 'shiller_gs10' if 'shiller_gs10' in data.columns else None
efw_feature = 'macro_efw' if 'macro_efw' in data.columns else None

# Plot a 2-year window to see alignment
start_zoom = '2005-01-01'
end_zoom = '2007-12-31'

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Daily data (GPR)
if gpr_feature:
    data.loc[start_zoom:end_zoom, gpr_feature].plot(
        ax=axes[0], marker='o', markersize=2, linewidth=0.5, color='steelblue'
    )
    axes[0].set_title('Daily Data: Geopolitical Risk (GPR)', fontweight='bold')
    axes[0].set_ylabel('GPR Index')
    axes[0].grid(alpha=0.3)

# Monthly data (Shiller yields) - aligned to daily
if shiller_feature:
    data.loc[start_zoom:end_zoom, shiller_feature].plot(
        ax=axes[1], marker='s', markersize=3, linewidth=1, color='darkgreen'
    )
    axes[1].set_title('Monthly → Daily: 10-Year Treasury Yield (Shiller)', fontweight='bold')
    axes[1].set_ylabel('Yield (%)')
    axes[1].grid(alpha=0.3)

# Annual data (EFW) - aligned to daily
if efw_feature:
    data.loc[start_zoom:end_zoom, efw_feature].plot(
        ax=axes[2], marker='^', markersize=4, linewidth=1, color='darkred'
    )
    axes[2].set_title('Annual → Daily: Economic Freedom (EFW) - Step Function', fontweight='bold')
    axes[2].set_ylabel('EFW Score')
    axes[2].grid(alpha=0.3)

# Format x-axis
for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))

plt.tight_layout()
plt.show()

print("\n📊 Notice the different interpolation patterns:")
print("  • Daily data (GPR): High-frequency variations")
print("  • Monthly data (Yields): Smooth daily interpolation")
print("  • Annual data (EFW): Step changes at year boundaries")

## 7. Asset Returns Overview

Extract and visualize returns for the 4 core assets.

In [ ]:
# Get asset list
assets = pipeline.get_asset_list()
print(f"Assets in {pipeline.mode} mode: {assets}")

# Note: Returns are constructed in feature engineering step
# Here we show the raw price data

if 'shiller_sp500' in data.columns:
    # Compute simple returns
    sp500_returns = data['shiller_sp500'].pct_change()
    
    # Plot cumulative returns
    cumulative_returns = (1 + sp500_returns).cumprod()
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # Returns
    sp500_returns.plot(ax=axes[0], color='steelblue', alpha=0.5, linewidth=0.5)
    axes[0].set_title('S&P 500 Daily Returns', fontweight='bold')
    axes[0].set_ylabel('Return')
    axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
    axes[0].grid(alpha=0.3)
    
    # Cumulative returns
    cumulative_returns.plot(ax=axes[1], color='darkgreen', linewidth=1.5)
    axes[1].set_title('S&P 500 Cumulative Returns (1985 = 1.0)', fontweight='bold')
    axes[1].set_ylabel('Cumulative Return')
    axes[1].set_yscale('log')
    axes[1].grid(alpha=0.3, which='both')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nS&P 500 Statistics (1985-2010):")
    print(f"  Annualized Return: {sp500_returns.mean() * 252 * 100:.2f}%")
    print(f"  Annualized Volatility: {sp500_returns.std() * np.sqrt(252) * 100:.2f}%")
    print(f"  Sharpe Ratio: {sp500_returns.mean() / sp500_returns.std() * np.sqrt(252):.2f}")
    print(f"  Maximum Drawdown: {(cumulative_returns / cumulative_returns.cummax() - 1).min() * 100:.2f}%")

## 8. Caching Performance

The `DataPipeline` automatically caches aligned data as compressed parquet files.

### Benefits:
- **20x faster** on repeated loads (0.1s vs 2s)
- **10x smaller** than CSV (compression)
- Automatic cache invalidation by date range

In [ ]:
import time

# Test cache performance
print("Testing cache performance...\n")

# First load (from source)
pipeline_test = DataPipeline(mode='basic', cache_dir='../data/cache_test')
start = time.time()
data1 = pipeline_test.load('2000-01-01', '2005-12-31', force_reload=True)
time_source = time.time() - start
print(f"Load from source: {time_source:.2f}s")

# Second load (from cache)
start = time.time()
data2 = pipeline_test.load('2000-01-01', '2005-12-31', force_reload=False)
time_cache = time.time() - start
print(f"Load from cache: {time_cache:.2f}s")

# Verify data is identical
assert data1.equals(data2), "Cached data doesn't match original!"
print(f"\n✓ Data verified identical")
print(f"Speedup: {time_source/time_cache:.1f}x faster")

## Summary

### Key Takeaways:

1. **DataPipeline Class**: Unified interface for loading 10+ heterogeneous sources
2. **Frequency Alignment**: Intelligent conversion (annual/monthly/daily → daily)
3. **Smart Caching**: 20x faster repeated loads with parquet compression
4. **Data Coverage**: 1985-2010 has excellent coverage for core features
5. **Two Modes**: 'basic' (20 features) for fast iteration, 'full' (90+ features) for comprehensive analysis

### Next Steps:

→ **Tutorial 2**: Feature Engineering - Transform raw data into regime identification features

### Resources:

- `src/core/data.py`: DataPipeline implementation
- `dataset/`: Raw data files
- `data/cache/`: Parquet cache directory